# Practice Lab : Linear Regression

In this lab i would be implementing a linear regression with one variabele to predict profits for a restaurant franchise.

_**Note**: this is a practice project ment to polish the fundamentals of aiml_

## 1 - Packages

run the cell below to import all the packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy 
import math 

## 2 - Problem Statement

Use the data provided to find the cities which may potentially give a restaurant business higher profits

## 3 - Dataset

In [34]:
data = "../../datasets/ex1data1.txt"
df = pd.read_csv(data)
x_train = df["population"]
y_train = df["profit"]

In [35]:
# print x_train
print("Type of x_train:",type(x_train))
print("The shape of x_train is:",x_train.shape)
print("First five elements of x_train are:\n" , x_train.head()) 



In [36]:
# print y_train
print("Type of y_train:",type(y_train))
print("The shape of y_train is:",y_train.shape)
print("First five elements of y_train are:\n", y_train.head())  

In [37]:
plt.scatter(x_train,y_train,marker="x",c="r")

plt.title("Profits vs Population per city")
plt.ylabel("Profit in $10000")
plt.xlabel("Population of city in 10000s")
plt.show()

## Linear Regression Model 

### Linear regression with one variable

The model function for lnear regression which maps x to y can be represented as

$$ f_{w,b}(x^{(i)}) = wx^{(i)} + b$$

### Cost Function of the Linear Regression Model

The cost function for linear regression $J(w,b)$ is defined as:

$$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2$$

### Compute Cost Function

In [38]:
def comp_cost(x,y,w,b):
    '''
    Computes the cost function for linear regression

    Args:
        x (array) : the feature part of the database
        y (array) : the label part of the database
        w,b : parameters of the model

    Returns:
        total_cost of using w,b as parameters for linear regression
    '''

    m = x.shape[0] #number of training exaples
    total_cost = 0 #retrun variable
    cost_sum = 0  

    for i in range(m):
        f_wb = x[i]*w + b
        cost_wb = (f_wb - y[i])**2

        cost_sum += cost_wb

    total_cost = (1/(2*m)) * cost_sum

    return total_cost

In [39]:
# Compute cost with some initial values for paramaters w, b # downloaded testcase
initial_w = 2
initial_b = 1

cost = comp_cost(x_train, y_train, initial_w, initial_b)
print(type(cost))
print(f'Cost at initial w: {cost:.3f}')


### Gradient Descent

The Gradient Descent algorithm is 

$$\begin{align*}& \text{repeat until convergence:} \; \lbrace \newline \; & \phantom {0000} b := b -  \alpha \frac{\partial J(w,b)}{\partial b} \newline       \; & \phantom {0000} w := w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{1}  \; & 
\newline & \rbrace\end{align*}$$

where, parameters $w, b$ are both updated simultaniously and where  
$$
\frac{\partial J(w,b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{2}
$$
$$
\frac{\partial J(w,b)}{\partial w}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) -y^{(i)})x^{(i)} \tag{3}
$$

In [40]:
def comp_gradient(x,y,w,b):
    '''
    Compute the gradient for linear regression

    Args:
        x (array) : the feature part of the database
        y (array) : the label part of the database
        w,b : parameters of the model

    returns:
        dj_db : the gradient of the cost wrt to b
        dj_dw : the gradient of the cost wrt to w
    '''

    m = x.shape[0]

    dj_db = 0
    dj_dw = 0

    for i in range(m):
        f_wbi = x[i]*w + b # Model function for that instance
        dj_dbi = (f_wbi - y[i]) #change in gradient for that instance wrt b
        dj_dwi = (f_wbi - y[i])*x[i] #change in gradient for that instance ert w

        dj_db += dj_dbi
        dj_dw += dj_dwi

    dj_db = dj_db/m
    dj_dw = dj_dw/m

    return dj_dw,dj_db

In [41]:
inital_w = 0
inital_b = 0

temp_dj_dw , temp_dj_db = comp_gradient(x_train,y_train,inital_w,inital_b)
print("Gradient at Inital w,b(zeros):",temp_dj_dw,temp_dj_db)

In [42]:
inital_w = 0.2
inital_b = 0.2

temp_dj_dw , temp_dj_db = comp_gradient(x_train,y_train,inital_w,inital_b)
print("Gradient at Inital w,b(zeros):",temp_dj_dw,temp_dj_db)

### Gradient Descent Algorithm (Coursera Lab referenced)

In [44]:
def gradient_descent(x, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn theta. Updates theta by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      x :    (ndarray): Shape (m,)
      y :    (ndarray): Shape (m,)
      w_in, b_in : (scalar) Initial values of parameters of the model
      cost_function: function to compute cost
      gradient_function: function to compute the gradient
      alpha : (float) Learning rate
      num_iters : (int) number of iterations to run gradient descent
    Returns
      w : (ndarray): Shape (1,) Updated values of parameters of the model after
          running gradient descent
      b : (scalar)                Updated value of parameter of the model after
          running gradient descent
    """
    
    # number of training examples
    m = len(x)
    
    # An array to store cost J and w's at each iteration — primarily for graphing later
    J_history = []
    w_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_dw, dj_db = gradient_function(x, y, w, b )  

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            cost =  cost_function(x, y, w, b)
            J_history.append(cost)

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters/10) == 0:
            w_history.append(w)
            print(f"Iteration {i:4}: Cost {float(J_history[-1]):8.2f}   ")
        
    return w, b, J_history, w_history #return w and J,w history for graphing

In [45]:
initial_w = 0.0
initial_b = 0.0

iterations = 2000
alpha = 0.01

w,b,_,_ = gradient_descent(x_train,y_train,inital_w,initial_b,comp_cost,comp_gradient,alpha,iterations)

print("w,b found by gradient descent:", w, b)

## Ploting the graph

In [46]:
m = x_train.shape[0]
predicted = np.zeros(m)

for i in range(m):
    predicted[i] = w * x_train[i] + b

In [47]:
plt.plot(x_train,predicted,c = "b")
plt.scatter(x_train,y_train,marker="x",c="r")

plt.title("Profits vs. Population per City")
plt.ylabel("Profit in $10000")
plt.xlabel("Population of a city in 10000")

In [48]:
pedict1 = 3.5 * w + b
pedict2 = 7 * w + b

print("For population = 350000 we predicted profit of $ : ", pedict1*10000)
print("For population = 7000000 we predicted profit of $ :",pedict2*10000)